# 02 - Training on Google Colab

Notebook này dùng để chạy training end-to-end trên Colab GPU cho project Vietnamese summarizer.

Mục tiêu chính:
- Chuẩn bị môi trường Colab và Drive.
- Clone repo, cài dependencies, kiểm tra GPU.
- Chuẩn bị data VietNews cho Phase 1.
- Chạy smoke test nhỏ trước khi train thật.
- Fine-tune Phase 1 trên VietNews.
- Evaluate model và lưu metrics/predictions.
- Nếu cần, chạy Phase 2 trên synthetic multi-mode data.

Trước khi chạy: vào `Runtime -> Change runtime type -> GPU`. T4 là đủ để bắt đầu.

## 0. Colab Runtime Checklist

Trước khi bấm Run All, kiểm tra nhanh:
- Runtime type đã chọn `GPU`.
- Google Drive đủ dung lượng cho checkpoint.
- Repo GitHub đã push bản mới nhất.
- Nếu Colab bị disconnect, chỉ cần chạy lại từ đầu; các thư mục `models`, `reports`, `data/processed` sẽ được giữ trong Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')

REPO_URL = 'https://github.com/CodeDaoVietNam/smart-vietnamese-summarizer.git'
REPO_BRANCH = 'develop'
REPO_DIR = Path('/content/smart-vietnamese-summarizer')
ARTIFACT_DIR = Path('/content/drive/MyDrive/smart-vietnamese-summarizer-artifacts')

for path in [ARTIFACT_DIR, ARTIFACT_DIR / 'models', ARTIFACT_DIR / 'reports', ARTIFACT_DIR / 'data' / 'processed']:
    path.mkdir(parents=True, exist_ok=True)

print('Repo URL:', REPO_URL)
print('Repo branch:', REPO_BRANCH)
print('Repo dir:', REPO_DIR)
print('Artifact dir:', ARTIFACT_DIR)

## 1. Clone Or Update Repo

Cell này clone repo từ branch `develop` vì code hiện tại của project đang nằm ở branch này. Nếu repo đã tồn tại trong session hiện tại, nó sẽ checkout `develop` rồi pull bản mới nhất.

In [ ]:
if REPO_DIR.exists():
    print('Repo already exists. Checking out target branch and pulling latest changes...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Current working directory:', Path.cwd())
print('Top-level files:')
print('\n'.join(sorted(p.name for p in Path.cwd().iterdir())[:30]))

required_paths = [Path('requirements.txt'), Path('configs/train_phase1.yaml'), Path('scripts/train.py')]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Repo root is not ready or the wrong branch was cloned. Missing: ' + ', '.join(missing)
    )
print('Repo sanity check passed.')

## 2. Persist Important Folders To Drive

Colab sẽ mất dữ liệu trong `/content` khi session tắt. Cell này nối `models`, `reports`, và `data/processed` sang Google Drive để checkpoint, metrics, và processed dataset không bị mất.

In [ ]:
def link_to_drive(relative_path: str) -> None:
    link_path = REPO_DIR / relative_path
    target_path = ARTIFACT_DIR / relative_path
    target_path.mkdir(parents=True, exist_ok=True)

    if link_path.is_symlink() and link_path.resolve() == target_path.resolve():
        print(f'{relative_path} already linked -> {target_path}')
        return

    if link_path.exists() or link_path.is_symlink():
        backup_path = REPO_DIR / f'{relative_path}.local_backup'
        if backup_path.exists():
            shutil.rmtree(backup_path) if backup_path.is_dir() else backup_path.unlink()
        link_path.rename(backup_path)
        print(f'Moved existing {relative_path} to {backup_path}')

    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(target_path, target_is_directory=True)
    print(f'Linked {relative_path} -> {target_path}')

for folder in ['models', 'reports', 'data/processed']:
    link_to_drive(folder)

## 3. Check GPU And Runtime

Nếu cell này không thấy GPU, quay lại `Runtime -> Change runtime type -> GPU` rồi chạy lại runtime.

In [ ]:
!nvidia-smi
!python --version
!pwd

## 4. Install Dependencies

Cell này có thể mất vài phút. Nếu Colab yêu cầu restart runtime sau khi cài package, restart rồi chạy lại từ đầu đến cell kiểm tra import.

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
import torch
import datasets
import transformers
import yaml

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('datasets:', datasets.__version__)
print('transformers:', transformers.__version__)

## 5. Optional Hugging Face Login

Không bắt buộc. Nếu bạn bị rate limit khi tải dataset/model, tạo token ở Hugging Face rồi dùng cell này.

In [ ]:
# Optional: uncomment nếu muốn login Hugging Face.
# from huggingface_hub import login
# login(token='PASTE_YOUR_HF_TOKEN_HERE')

## 6. Create Colab Configs

Mặc định notebook dùng cấu hình Colab thực tế: train một phần dataset để tiết kiệm thời gian. Khi bạn muốn train full dataset, đổi `RUN_FULL_DATASET = True` rồi chạy lại cell này và các cell prepare/train bên dưới.

Gợi ý:
- `RUN_FULL_DATASET = False`: phù hợp để demo, kiểm pipeline, làm báo cáo tuần 1.
- `RUN_FULL_DATASET = True`: train thật lâu hơn nhiều, cần session ổn định.

In [ ]:
import yaml
from copy import deepcopy
from pathlib import Path

RUN_FULL_DATASET = False

TRAIN_LIMIT = None if RUN_FULL_DATASET else 30000
VALIDATION_LIMIT = None if RUN_FULL_DATASET else 3000
TEST_LIMIT = None if RUN_FULL_DATASET else 3000
EVAL_LIMIT = None if RUN_FULL_DATASET else 200
EPOCHS = 3 if RUN_FULL_DATASET else 1

with open('configs/train_phase1.yaml', 'r', encoding='utf-8') as file:
    phase1_cfg = yaml.safe_load(file)

colab_cfg = deepcopy(phase1_cfg)
colab_cfg['dataset']['train_file'] = 'data/processed/train_colab.jsonl'
colab_cfg['dataset']['validation_file'] = 'data/processed/validation_colab.jsonl'
colab_cfg['dataset']['test_file'] = 'data/processed/test_colab.jsonl'
colab_cfg['dataset']['max_samples'] = {
    'train': TRAIN_LIMIT,
    'validation': VALIDATION_LIMIT,
    'test': TEST_LIMIT,
}
colab_cfg['model']['output_dir'] = 'models/vit5-summarizer-v1'
colab_cfg['training']['epochs'] = EPOCHS
colab_cfg['training']['logging_steps'] = 25 if not RUN_FULL_DATASET else phase1_cfg['training']['logging_steps']

colab_config_path = Path('configs/_colab_train_phase1.yaml')
with colab_config_path.open('w', encoding='utf-8') as file:
    yaml.safe_dump(colab_cfg, file, allow_unicode=True, sort_keys=False)

print(colab_config_path.read_text(encoding='utf-8'))

## 7. Smoke Test Config

Smoke test dùng dataset rất nhỏ để bắt lỗi import, tokenizer, GPU, Trainer trước khi train thật. Nên chạy bước này một lần.

In [ ]:
smoke_cfg = deepcopy(phase1_cfg)
smoke_cfg['dataset']['train_file'] = 'data/processed/smoke_train.jsonl'
smoke_cfg['dataset']['validation_file'] = 'data/processed/smoke_validation.jsonl'
smoke_cfg['dataset']['test_file'] = 'data/processed/smoke_test.jsonl'
smoke_cfg['dataset']['max_samples'] = {'train': 64, 'validation': 16, 'test': 16}
smoke_cfg['model']['output_dir'] = 'models/vit5-smoke-test'
smoke_cfg['training']['epochs'] = 1
smoke_cfg['training']['logging_steps'] = 2
smoke_cfg['training']['save_total_limit'] = 1

smoke_config_path = Path('configs/_colab_train_phase1_smoke.yaml')
with smoke_config_path.open('w', encoding='utf-8') as file:
    yaml.safe_dump(smoke_cfg, file, allow_unicode=True, sort_keys=False)

print(smoke_config_path.read_text(encoding='utf-8'))

## 8. Run Smoke Test

Nếu bước này pass, pipeline training đã ổn. Nếu lỗi ở đây, sửa lỗi trước khi chạy training lớn.

In [ ]:
!python scripts/prepare_data.py --config configs/_colab_train_phase1_smoke.yaml
!python scripts/train.py --config configs/_colab_train_phase1_smoke.yaml

## 9. Prepare Phase 1 Data

Cell này tải VietNews từ Hugging Face và lưu JSONL đã normalize vào `data/processed`. Nếu bạn dùng `RUN_FULL_DATASET = False`, nó chỉ lấy một phần dataset theo config Colab.

In [ ]:
!python scripts/prepare_data.py --config configs/_colab_train_phase1.yaml

In [ ]:
import json
from pathlib import Path

for split_file in ['train_colab.jsonl', 'validation_colab.jsonl', 'test_colab.jsonl']:
    path = Path('data/processed') / split_file
    count = sum(1 for _ in path.open('r', encoding='utf-8'))
    print(f'{split_file}: {count:,} rows')

sample = json.loads(Path('data/processed/test_colab.jsonl').read_text(encoding='utf-8').splitlines()[0])
print('\nSample keys:', sample.keys())
print('\nDocument preview:', sample['document'][:500])
print('\nReference summary:', sample['summary'])

## 10. Baseline Prediction Before Training

`configs/app.yaml` đang trỏ đến model phase 2, nhưng nếu chưa có checkpoint thì code tự fallback về `VietAI/vit5-base`. Đây là baseline trước khi fine-tune.

In [ ]:
sample_input_path = Path('reports/examples/colab_sample_input.txt')
sample_input_path.parent.mkdir(parents=True, exist_ok=True)
sample_input_path.write_text(sample['document'], encoding='utf-8')

!python scripts/predict.py --text-file reports/examples/colab_sample_input.txt --mode concise --length medium --config configs/app.yaml

## 11. Train Phase 1

Phase 1 fine-tune ViT5 trên VietNews. Với config Colab mặc định, model được lưu vào `models/vit5-summarizer-v1`, và vì thư mục `models` đã link sang Drive nên checkpoint được giữ lại.

Nếu bị CUDA out of memory:
- Giảm `per_device_train_batch_size` từ `2` xuống `1` trong config.
- Hoặc giảm `max_source_length` từ `512` xuống `384`.
- Hoặc giữ batch size nhưng giảm `TRAIN_LIMIT`.

In [ ]:
!python scripts/train.py --config configs/_colab_train_phase1.yaml

## 12. Plot Phase 1 Training Loss

Script training lưu log vào `reports/metrics/training_log.json`. Cell này vẽ nhanh loss curve để đưa vào report.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

log_path = Path('reports/metrics/training_log.json')
payload = json.loads(log_path.read_text(encoding='utf-8'))
history = pd.DataFrame(payload['log_history'])
display(history.tail())

loss_rows = history.dropna(subset=['loss']) if 'loss' in history.columns else pd.DataFrame()
if not loss_rows.empty:
    plt.figure(figsize=(10, 4))
    plt.plot(loss_rows['step'], loss_rows['loss'], marker='o')
    plt.title('Phase 1 Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    figure_path = Path('reports/figures/phase1_training_loss.png')
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(figure_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved figure:', figure_path)

## 13. Evaluate Phase 1

Cell này tạo config evaluate tạm cho model `v1`, chạy ROUGE trên test set Colab, rồi lưu metrics/predictions vào `reports`.

In [ ]:
with open('configs/eval.yaml', 'r', encoding='utf-8') as file:
    eval_cfg = yaml.safe_load(file)

eval_phase1_cfg = deepcopy(eval_cfg)
eval_phase1_cfg['model']['name'] = 'models/vit5-summarizer-v1'
eval_phase1_cfg['model']['fallback_name'] = 'VietAI/vit5-base'
eval_phase1_cfg['dataset']['test_file'] = 'data/processed/test_colab.jsonl'
eval_phase1_cfg['dataset']['max_samples'] = EVAL_LIMIT
eval_phase1_cfg['outputs']['metrics_file'] = 'reports/metrics/eval_phase1.json'
eval_phase1_cfg['outputs']['predictions_file'] = 'reports/examples/test_predictions_phase1.jsonl'

eval_phase1_path = Path('configs/_colab_eval_phase1.yaml')
with eval_phase1_path.open('w', encoding='utf-8') as file:
    yaml.safe_dump(eval_phase1_cfg, file, allow_unicode=True, sort_keys=False)

!python scripts/evaluate.py --config configs/_colab_eval_phase1.yaml
!cat reports/metrics/eval_phase1.json

In [ ]:
prediction_path = Path('reports/examples/test_predictions_phase1.jsonl')
rows = [json.loads(line) for line in prediction_path.read_text(encoding='utf-8').splitlines()[:3]]

for i, row in enumerate(rows, start=1):
    print('=' * 90)
    print('Sample', i)
    print('\nDocument:', row['document'][:700])
    print('\nReference:', row['reference'])
    print('\nPrediction:', row['prediction'])
    print('\nError tags:', row['error_tags'])

## 14. Phase 1 Prediction After Training

Cell này dùng checkpoint `v1` để sinh summary cho sample ban đầu. Đây là nơi so sánh nhanh baseline vs fine-tuned model.

In [ ]:
app_phase1_cfg = yaml.safe_load(Path('configs/app.yaml').read_text(encoding='utf-8'))
app_phase1_cfg['model']['name'] = 'models/vit5-summarizer-v1'
app_phase1_cfg['model']['fallback_name'] = 'VietAI/vit5-base'
app_phase1_path = Path('configs/_colab_app_phase1.yaml')
app_phase1_path.write_text(yaml.safe_dump(app_phase1_cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

!python scripts/predict.py --text-file reports/examples/colab_sample_input.txt --mode concise --length medium --config configs/_colab_app_phase1.yaml

## 15. Generate And Inspect Synthetic Data For Phase 2

Phase 2 giúp model học các output mode như `bullet`, `action_items`, `study_notes`. Dataset synthetic hiện là starter set nhỏ, đủ để kiểm pipeline và minh họa controllable summarization.

In [ ]:
!python scripts/generate_synthetic.py

for file_name in ['data/synthetic/train.jsonl', 'data/synthetic/validation.jsonl']:
    path = Path(file_name)
    print('\n', file_name)
    for line in path.read_text(encoding='utf-8').splitlines():
        print(json.loads(line))

## 16. Train Phase 2

Chạy cell này sau khi `models/vit5-summarizer-v1` đã tồn tại. Phase 2 load checkpoint v1 rồi lưu checkpoint mới vào `models/vit5-summarizer-v2`.

In [ ]:
!test -d models/vit5-summarizer-v1 && echo 'Phase 1 checkpoint exists.'
!python scripts/train_phase2.py --config configs/train_phase2.yaml

## 17. Evaluate Phase 2

Config mặc định `configs/eval.yaml` đang trỏ tới `models/vit5-summarizer-v2`. Cell này evaluate model phase 2 trên cùng test set Colab để có số so sánh.

In [ ]:
eval_phase2_cfg = deepcopy(eval_cfg)
eval_phase2_cfg['model']['name'] = 'models/vit5-summarizer-v2'
eval_phase2_cfg['model']['fallback_name'] = 'models/vit5-summarizer-v1'
eval_phase2_cfg['dataset']['test_file'] = 'data/processed/test_colab.jsonl'
eval_phase2_cfg['dataset']['max_samples'] = EVAL_LIMIT
eval_phase2_cfg['outputs']['metrics_file'] = 'reports/metrics/eval_phase2.json'
eval_phase2_cfg['outputs']['predictions_file'] = 'reports/examples/test_predictions_phase2.jsonl'

eval_phase2_path = Path('configs/_colab_eval_phase2.yaml')
with eval_phase2_path.open('w', encoding='utf-8') as file:
    yaml.safe_dump(eval_phase2_cfg, file, allow_unicode=True, sort_keys=False)

!python scripts/evaluate.py --config configs/_colab_eval_phase2.yaml
!cat reports/metrics/eval_phase2.json

## 18. Compare Modes After Phase 2

Một input, bốn mode. Đây là phần rất hay để đưa vào demo/report vì nó cho thấy product behavior, không chỉ metric.

In [ ]:
for mode in ['concise', 'bullet', 'action_items', 'study_notes']:
    print('\n' + '=' * 90)
    print('MODE:', mode)
    subprocess.run(
        [
            'python',
            'scripts/predict.py',
            '--text-file',
            'reports/examples/colab_sample_input.txt',
            '--mode',
            mode,
            '--length',
            'medium',
            '--config',
            'configs/app.yaml',
        ],
        check=True,
    )

## 19. Collect Final Artifacts

Vì `models` và `reports` đã link sang Drive, artifact đã được lưu tự động. Cell này chỉ in lại vị trí để bạn dễ mở từ Drive.

In [ ]:
print('Artifacts in Drive:')
print(ARTIFACT_DIR)

print('\nModels:')
!find models -maxdepth 2 -type f | head -30

print('\nReports:')
!find reports -maxdepth 3 -type f | sort

## 20. What To Write In The Report

Sau khi chạy xong notebook, ghi lại các ý này vào report:
- Dataset dùng để train: VietNews, split train/validation/test sau khi giới hạn Colab.
- Model base: `VietAI/vit5-base`.
- Phase 1 config: batch size, gradient accumulation, max source length, max target length, epochs.
- Smoke test pass hay lỗi gì.
- Phase 1 training loss và validation loss.
- ROUGE Phase 1 và Phase 2 nếu có.
- 3 sample prediction: document preview, reference, prediction, nhận xét lỗi.
- Với Phase 2, so sánh output giữa `concise`, `bullet`, `action_items`, `study_notes`.

Kết luận mong muốn: model đã chạy được pipeline fine-tuning tiếng Việt end-to-end, Phase 1 học summarization core, Phase 2 bắt đầu học controllable modes.

## Troubleshooting Quick Notes

- `No module named smart_summarizer`: kiểm tra đang ở đúng repo root bằng `!pwd`, rồi chạy lại script từ root.
- `CUDA out of memory`: giảm batch size hoặc `max_source_length` trong config Colab.
- `HF Hub unauthenticated`: có thể bỏ qua, hoặc login Hugging Face token.
- Colab disconnect: chạy lại notebook; checkpoint và processed data vẫn nằm trong Drive.
- `models/vit5-summarizer-v1` không tồn tại trước Phase 2: chạy xong Phase 1 trước.
- Metrics quá thấp: kiểm tra sample prediction, không chỉ nhìn ROUGE. Với summarization tiếng Việt, qualitative review rất quan trọng.